# Modal Setup and the Specialist Agent

## Order

1. Modal.com and SpecialistAgent  
2. RAG, FrontierAgent, EnsembleAgent  
3. ScannerAgent, MessagingAgent  
4. AutonomousPlanningAgent and DealAgentFramework  
5. Gradio finale

In [ ]:
import locale
import os

import modal
from dotenv import load_dotenv

from utils.items import Wine
from utils.preprocessor import TextAssembler

load_dotenv(override=True)

In [ ]:
# Confirm the local machine can output UTF-8 (should print 'UTF-8')
print(locale.getpreferredencoding())

In [ ]:
os.environ["PYTHONIOENCODING"] = "utf-8"

# Setting up Modal tokens

First visit https://modal.com and sign up. Click the Avatar menu (top right) -> Settings ->
API Tokens -> New Token.

You'll be given something like:

`modal token set --token-id ak-somethinghere --token-secret as-somethinghere`

Since this project uses uv, run it as:

`uv run modal token set --token-id ak-somethinghere --token-secret as-somethinghere`

Alternatively, add the two keys directly to your `.env` file:

```
MODAL_TOKEN_ID=ak-...
MODAL_TOKEN_SECRET=as-...
```

then rerun `load_dotenv(override=True)`.

In [ ]:
from hello import app, hello, hello_europe

In [ ]:
with app.run():
    reply = hello.local()
reply

In [ ]:
with app.run():
    reply = hello.remote()
reply

In [ ]:
with app.run():
    reply = hello_europe.remote()
reply

# Setting your HuggingFace Token as a Modal secret

Secrets in Modal have a **name**, then a KEY and VALUE inside. We need:

Name: `huggingface-secret`  
Key: `HF_TOKEN`  
Value: `hf_...`

1. Go to modal.com -> Secrets -> Create new secret -> Hugging Face
2. Name it exactly `huggingface-secret` (that's how it's referenced in code)
3. Fill in HF_TOKEN with your actual token
4. Done — the Llama base model is gated, so this token is required to load it

In [ ]:
from llama import app, generate

In [ ]:
with modal.enable_output():
    with app.run():
        result = generate.remote("A wine with dark fruit and firm tannins is typically")
result

# Building a sample wine and preprocessing it

The specialist model expects text assembled by `utils.preprocessor.TextAssembler` —
the same deterministic assembly used at training time. No LLM rewriting step here.

In [ ]:
sample = Wine(
    title="2021 Reserve Cabernet Sauvignon",
    points=91,
    price=35.0,
    country="US",
    province="California",
    region="Napa Valley",
    variety="Cabernet Sauvignon",
    winery="Example Winery",
    full=(
        "Dark and structured, with blackberry and cassis notes framed by firm "
        "tannins and a long, cedar-tinged finish."
    ),
)

text = TextAssembler.assemble(sample)
print(text)

In [ ]:
from specialist_ephemeral import app, estimate_vfm

In [ ]:
with modal.enable_output():
    with app.run():
        result = estimate_vfm.remote(text)
result

## Transitioning from Ephemeral Apps to Deployed Apps

`uv run modal deploy -m xxx` packages the code as a Deployed App — this is how the
specialist model gets served behind a stable name that the agent layer can call
repeatedly without redeploying.

Both `specialist_service.py` and `specialist_service2.py` reference
`modal.Secret.from_name("huggingface-secret")` — rename if your Modal secret uses a
different name.

In [ ]:
# You can also run this from a terminal: uv run modal deploy -m specialist_service
!uv run modal deploy -m specialist_service

In [ ]:
specialist = modal.Function.from_name("wine-vfm-specialist-service", "estimate_vfm")
specialist.remote(text)

In [ ]:
# You can also run this from a terminal: uv run modal deploy -m specialist_service2
!uv run modal deploy -m specialist_service2

In [ ]:
WineSpecialist = modal.Cls.from_name("wine-vfm-specialist-service", "WineSpecialist")
specialist = WineSpecialist()
reply = specialist.estimate_vfm.remote(text)
print(reply)

In [ ]:
reply = specialist.estimate_vfm.remote(text)
print(reply)

# Optional: keeping Modal warm

The first call to `WineSpecialist` can take minutes to build the container. After that
it should be ~30s if cold, ~2s if warm. To keep a container alive continuously, set
`MIN_CONTAINERS = 1` in `specialist_service2.py` — this consumes Modal credits
continuously, so only do this deliberately.

To extend the warm window temporarily instead:

```python
WineSpecialist = modal.Cls.from_name("wine-vfm-specialist-service", "WineSpecialist")
specialist = WineSpecialist()
specialist.update_autoscaler(scaledown_window=1200)  # stay warm 20 min
```

## The SpecialistAgent

Wraps the deployed `WineSpecialist` class behind a stable agent interface.

In [ ]:
import logging

root = logging.getLogger()
root.setLevel(logging.INFO)

In [ ]:
from agents.specialist_agent import SpecialistAgent

In [ ]:
agent = SpecialistAgent()

In [ ]:
agent.estimate(text)